# LMM (Titans): Mortality Prediction on MIMIC-III

This notebook demonstrates the **LMM** (Neural Long-term Memory) model for mortality prediction on MIMIC-III.

**Paper**: Ali Behrouz et al. "Titans: Learning to Memorize at Test Time." arXiv 2501.00663, 2025.

LMM uses surprise-based gradient updates with momentum to preferentially memorize unexpected inputs. For EHR data, this means rare but clinically significant events (a sudden lab spike, a new critical diagnosis) are stored more strongly in memory than routine visits.

This notebook shows two configurations:
1. **LMM standalone** — LMM as a direct sequence encoder (like RNN)
2. **GRASP + LMM** — LMM as the backbone inside GRASP's patient clustering framework

### Compute Tracking

This notebook tracks energy consumption, GPU utilization, and CO2 emissions for each training run using `codecarbon` and `pynvml`. This data supports research into the environmental cost of ML experiments and helps inform datacenter policy.

## Step 1: Load the MIMIC-III Dataset

We load the MIMIC-III dataset using PyHealth's `MIMIC3Dataset` class.

**Two configurations:**
- **Local (default):** Set `dev=True` with synthetic GCS data for quick testing
- **H200 / full run:** Set `USE_LOCAL_DATA = True` below and point `LOCAL_ROOT` to your MIMIC-III CSV directory

In [3]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

# ── Configuration ─────────────────────────────────────────
# Set USE_LOCAL_DATA = True on the H200 where MIMIC-III CSVs live
USE_LOCAL_DATA = True
LOCAL_ROOT = "/home/lolowo2"           # directory with ADMISSIONS.csv.gz, etc.
LOCAL_CACHE = "/home/lolowo2/tmp/pyhealth_cache"
# ──────────────────────────────────────────────────────────

if USE_LOCAL_DATA:
    base_dataset = MIMIC3Dataset(
        root=LOCAL_ROOT,
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=LOCAL_CACHE,
        dev=False,  # full dataset
    )
else:
    base_dataset = MIMIC3Dataset(
        root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
        tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
        cache_dir=tempfile.TemporaryDirectory().name,
        dev=True,  # small synthetic subset
    )

base_dataset.stats()

No config path provided, using default config
Initializing mimic3 dataset from /home/lolowo2 (dev mode: False)
Using provided cache_dir: /home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c
No cached event dataframe found. Creating: /home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet
Scanning table: patients from /home/lolowo2/PATIENTS.csv.gz
Scanning table: admissions from /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: icustays from /home/lolowo2/ICUSTAYS.csv.gz
Scanning table: diagnoses_icd from /home/lolowo2/DIAGNOSES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: procedures_icd from /home/lolowo2/PROCEDURES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: prescriptions from /home/lolowo2/PRESCRIPTIONS.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Caching event dataframe to /home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_

## Step 2: Define the Mortality Prediction Task

The `MortalityPredictionMIMIC3` task extracts samples from the raw EHR data:
- Extracts diagnosis codes (ICD-9), procedure codes, and drug information from each visit
- Creates binary labels based on in-hospital mortality
- Filters out visits without sufficient clinical codes

In [4]:
from pyhealth.tasks import MortalityPredictionMIMIC3

task = MortalityPredictionMIMIC3()
samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/task_df.ld, samples=/home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 46520 patients. (Polars threads: 10)


  0%|          | 0/46520 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 46520/46520 [01:04<00:00, 717.15it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

  0%|          | 0/9583 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 9583/9583 [00:00<00:00, 16077.30it/s]

Worker 0 finished processing samples.


Cached processed samples to /home/lolowo2/tmp/pyhealth_cache/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_187a839c-4f67-585a-bbb3-355429e27594/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Generated 9583 samples

Input schema: {'conditions': 'sequence', 'procedures': 'sequence', 'drugs': 'sequence'}
Output schema: {'mortality': 'binary'}


## Step 3: Split Data and Create Loaders

We split by patient to avoid data leakage, then create batched data loaders.

In [5]:
from pyhealth.datasets import split_by_patient, get_dataloader

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1]
)

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 7661
Validation samples: 938
Test samples: 984


## Compute Tracking Setup

We log hardware specs, energy consumption, GPU power/utilization, and CO2 emissions.

Install dependencies if needed:
```bash
pip install codecarbon pynvml
```

**Metrics tracked per training run:**

| Metric | What it measures | Unit |
|---|---|---|
| Wall time | Real elapsed clock time | seconds |
| Energy consumed | Electricity drawn by GPU | kWh |
| CO2 emissions | Energy × grid carbon intensity | kg CO2eq |
| Peak / avg GPU power | How hard the chip works | Watts |
| GPU utilization | % time GPU cores are computing | % |
| GPU memory | VRAM allocated vs available | GB |
| Energy per epoch | Diminishing returns indicator | kWh |
| Energy per sample | Unit cost for deployment | Joules |
| Batch throughput | Samples processed per second | samples/s |

In [7]:
import time
import platform
import torch

# ── Hardware Info ──────────────────────────────────────────
print("=" * 60)
print("Hardware Information")
print("=" * 60)
print(f"Platform:     {platform.system()} {platform.machine()}")
print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:          {gpu_name}")
    print(f"GPU Memory:   {gpu_mem_gb:.1f} GB")
    print(f"CUDA:         {torch.version.cuda}")
else:
    print("GPU:          None (CPU only)")
    print("  Note: Compute metrics will be limited without GPU")

# ── GPU Monitoring Helper ─────────────────────────────────
gpu_available = torch.cuda.is_available()
nvml_available = False

if gpu_available:
    try:
        import pynvml
        pynvml.nvmlInit()
        nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        nvml_available = True
        print(f"\npynvml:       initialized (live GPU monitoring enabled)")
    except (ImportError, Exception) as e:
        print(f"\npynvml:       not available ({e})")
        print("  Install with: pip install pynvml")

# ── Carbon Tracking Helper ────────────────────────────────
codecarbon_available = False
try:
    from codecarbon import EmissionsTracker
    codecarbon_available = True
    print(f"codecarbon:   available (CO2 tracking enabled)")
except ImportError:
    print(f"codecarbon:   not available")
    print("  Install with: pip install codecarbon")


class ComputeTracker:
    """Tracks energy, GPU metrics, and timing for a training run."""

    def __init__(self, name, num_samples, num_epochs):
        self.name = name
        self.num_samples = num_samples
        self.num_epochs = num_epochs
        self.power_readings = []
        self.util_readings = []
        self.mem_readings = []

    def start(self):
        self.start_time = time.time()
        if gpu_available:
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        if codecarbon_available:
            self.tracker = EmissionsTracker(
                project_name=self.name,
                log_level="error",
                save_to_file=False,
            )
            self.tracker.start()
        self._sample_gpu()
        return self

    def _sample_gpu(self):
        if nvml_available:
            power = pynvml.nvmlDeviceGetPowerUsage(nvml_handle) / 1000  # mW -> W
            util = pynvml.nvmlDeviceGetUtilizationRates(nvml_handle).gpu
            mem_info = pynvml.nvmlDeviceGetMemoryInfo(nvml_handle)
            self.power_readings.append(power)
            self.util_readings.append(util)
            self.mem_readings.append(mem_info.used / 1e9)

    def sample(self):
        """Call periodically during training to collect GPU readings."""
        self._sample_gpu()

    def stop(self):
        if gpu_available:
            torch.cuda.synchronize()
        self.wall_time = time.time() - self.start_time
        self._sample_gpu()

        self.emissions_data = None
        if codecarbon_available:
            self.tracker.stop()
            self.emissions_data = self.tracker.final_emissions_data

        return self.report()

    def report(self):
        r = {"name": self.name}
        r["wall_time_s"] = self.wall_time
        r["wall_time_min"] = self.wall_time / 60
        r["num_epochs"] = self.num_epochs
        r["num_samples"] = self.num_samples

        # Throughput
        total_samples_processed = self.num_samples * self.num_epochs
        r["batch_throughput_sps"] = total_samples_processed / self.wall_time
        r["time_per_epoch_s"] = self.wall_time / max(self.num_epochs, 1)

        # Energy (from codecarbon)
        if self.emissions_data:
            r["energy_kwh"] = self.emissions_data.energy_consumed
            r["co2_kg"] = self.emissions_data.emissions
            r["energy_per_epoch_kwh"] = r["energy_kwh"] / max(self.num_epochs, 1)
            r["energy_per_sample_j"] = (r["energy_kwh"] * 3.6e6) / max(total_samples_processed, 1)
        else:
            r["energy_kwh"] = None
            r["co2_kg"] = None

        # GPU metrics (from pynvml)
        if self.power_readings:
            r["peak_power_w"] = max(self.power_readings)
            r["avg_power_w"] = sum(self.power_readings) / len(self.power_readings)
            r["avg_gpu_util_pct"] = sum(self.util_readings) / len(self.util_readings)
            r["peak_gpu_mem_gb"] = max(self.mem_readings)
        elif gpu_available:
            r["peak_gpu_mem_gb"] = torch.cuda.max_memory_allocated() / 1e9

        return r


def print_report(r):
    print(f"\n{'=' * 50}")
    print(f"Compute Report: {r['name']}")
    print(f"{'=' * 50}")
    print(f"Wall time:          {r['wall_time_s']:.1f}s ({r['wall_time_min']:.2f} min)")
    print(f"Epochs:             {r['num_epochs']}")
    print(f"Time per epoch:     {r['time_per_epoch_s']:.2f}s")
    print(f"Throughput:         {r['batch_throughput_sps']:.1f} samples/s")

    if r.get("energy_kwh") is not None:
        print(f"\nEnergy consumed:    {r['energy_kwh']:.6f} kWh")
        print(f"CO2 emissions:      {r['co2_kg']:.6f} kg CO2eq")
        print(f"Energy per epoch:   {r['energy_per_epoch_kwh']:.6f} kWh")
        print(f"Energy per sample:  {r['energy_per_sample_j']:.4f} J")

    if r.get("peak_power_w"):
        print(f"\nPeak GPU power:     {r['peak_power_w']:.0f} W")
        print(f"Avg GPU power:      {r['avg_power_w']:.0f} W")
        print(f"Avg GPU utilization: {r['avg_gpu_util_pct']:.0f}%")

    if r.get("peak_gpu_mem_gb"):
        total_gb = gpu_mem_gb if gpu_available else 0
        used = r["peak_gpu_mem_gb"]
        print(f"Peak GPU memory:    {used:.2f} GB / {total_gb:.0f} GB ({used/max(total_gb,1)*100:.0f}%)")

    print(f"{'=' * 50}")


print("\nComputeTracker ready.")

Hardware Information
Platform:     Linux x86_64
Python:       3.13.11
PyTorch:      2.7.1+cu126
GPU:          NVIDIA H200 NVL
GPU Memory:   150.1 GB
CUDA:         12.6

pynvml:       initialized (live GPU monitoring enabled)
codecarbon:   available (CO2 tracking enabled)

ComputeTracker ready.


## Step 4: LMM Standalone Model

First, we train the LMM model as a standalone sequence encoder — the same role as RNN, but using surprise-based memory instead of fixed gates.

### Key Parameters:
- `embedding_dim`: Dimension of code embeddings
- `hidden_dim`: Dimension of the memory's hidden state
- `memory_depth`: Number of layers in the memory MLP (default: 2)
- `use_momentum`: Whether surprise includes momentum from past timesteps (default: True)
- `use_weight_decay`: Whether memory undergoes adaptive forgetting (default: True)

In [8]:
from pyhealth.models import LMM

# Using best hyperparams from GRASP GRU sweep:
# emb=32, hid=32, cluster=8, lr=5e-4, wd=1e-4
lmm_model = LMM(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    memory_depth=2,
)

print(f"LMM model: {sum(p.numel() for p in lmm_model.parameters()):,} parameters")

/home/lolowo2/git/PyHealth_titans_lmm/pyhealth/metrics/calibration.py:122: SyntaxWarning: invalid escape sequence '\c'
  accuracy of 1. Thus, the ECE is :math:`\\frac{1}{3} \cdot 0.49 + \\frac{2}{3}\cdot 0.3=0.3633`.


LMM model: 307,306 parameters


In [9]:
from pyhealth.trainer import Trainer

NUM_EPOCHS = 50

lmm_trainer = Trainer(
    model=lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

# Start compute tracking
lmm_tracker = ComputeTracker(
    "LMM-standalone", len(train_dataset), NUM_EPOCHS,
).start()

lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 5e-4, "weight_decay": 1e-4},
)

lmm_compute = lmm_tracker.stop()
print_report(lmm_compute)

LMM(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 32, padding_idx=0)
    (procedures): Embedding(1304, 32, padding_idx=0)
    (drugs): Embedding(3495, 32, padding_idx=0)
  ))
  (lmm): ModuleDict(
    (conditions): LMMLayer(
      (proj_k): Linear(in_features=32, out_features=32, bias=True)
      (proj_v): Linear(in_features=32, out_features=32, bias=True)
      (proj_q): Linear(in_features=32, out_features=32, bias=True)
      (memory): Sequential(
        (0): Linear(in_features=32, out_features=64, bias=True)
        (1): SiLU()
        (2): Linear(in_features=64, out_features=32, bias=True)
      )
      (gate_theta): Linear(in_features=32, out_features=1, bias=True)
      (gate_eta): Linear(in_features=32, out_features=1, bias=True)
      (gate_alpha): Linear(in_features=32, out_features=1, bias=True)
      (dropout_layer): Dropout(p=0.5, inplace=False)
    )
    (procedures): LMMLayer(
      (proj_k): Linear(in_features=32, out

[codecarbon WARNING @ 05:56:52] Multiple instances of codecarbon are allowed to run at the same time.


Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.0005, 'weight_decay': 0.0001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x732472615310>
Monitor: roc_auc
Monitor criterion: max
Epochs: 50
Patience: None



Epoch 0 / 50:   0%|          | 0/240 [00:00<?, ?it/s]

--- Train epoch-0, step-240 ---
loss: 0.6314


...

Epoch 49 / 50:   0%|          | 0/240 [00:00<?, ?it/s]

--- Train epoch-49, step-12000 ---
loss: 0.3603


Evaluation: 100%|██████████| 30/30 [00:10<00:00,  2.92it/s]

--- Eval epoch-49, step-12000 ---
roc_auc: 0.4801
pr_auc: 0.1250
accuracy: 0.8785
f1: 0.0000
loss: 0.3813
New best roc_auc score (0.4801) at epoch-49, step-12000
Loaded best model



Compute Report: LMM-standalone
Wall time:          6231.5s (103.86 min)
Epochs:             50
Time per epoch:     124.63s
Throughput:         61.5 samples/s

Energy consumed:    2.678351 kWh
CO2 emissions:      0.985654 kg CO2eq
Energy per epoch:   0.053567 kWh
Energy per sample:  25.1718 J

Peak GPU power:     93 W
Avg GPU power:      92 W
Avg GPU utilization: 11%
Peak GPU memory:    1.72 GB / 150 GB (1%)


In [10]:
lmm_results = lmm_trainer.evaluate(test_dataloader)

print("=" * 50)
print("LMM Standalone — Test Set Performance")
print("=" * 50)
for metric, value in lmm_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 31/31 [00:12<00:00,  2.54it/s]

LMM Standalone — Test Set Performance
roc_auc: 0.5212
pr_auc: 0.1248
accuracy: 0.8852
f1: 0.0000
loss: 0.3543


## Step 5: GRASP + LMM

Now we use LMM as the backbone inside GRASP. GRASP adds patient clustering via k-means, refines cluster representations with a GCN, and blends cluster knowledge back via a learned gate.

This tests whether surprise-based memory (LMM) and cross-patient knowledge (GRASP) are complementary:
- **LMM**: "What should I remember from *this* patient's history?"
- **GRASP**: "What can I learn from *similar* patients?"

In [11]:
from pyhealth.models import GRASP

grasp_lmm_model = GRASP(
    dataset=samples,
    embedding_dim=32,
    hidden_dim=32,
    cluster_num=8 if USE_LOCAL_DATA else 2,  # 8 for full data, 2 for dev
    block="LMM",
)

print(f"GRASP+LMM model: {sum(p.numel() for p in grasp_lmm_model.parameters()):,} parameters")

GRASP+LMM model: 314,032 parameters


In [12]:
grasp_lmm_trainer = Trainer(
    model=grasp_lmm_model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

# Start compute tracking
grasp_tracker = ComputeTracker(
    "GRASP+LMM", len(train_dataset), NUM_EPOCHS,
).start()

grasp_lmm_trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=NUM_EPOCHS,
    monitor="roc_auc",
    optimizer_params={"lr": 5e-4, "weight_decay": 1e-4},
)

grasp_compute = grasp_tracker.stop()
print_report(grasp_compute)

GRASP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(4102, 32, padding_idx=0)
    (procedures): Embedding(1304, 32, padding_idx=0)
    (drugs): Embedding(3495, 32, padding_idx=0)
  ))
  (grasp): ModuleDict(
    (conditions): GRASPLayer(
      (backbone): LMMLayer(
        (proj_k): Linear(in_features=32, out_features=32, bias=True)
        (proj_v): Linear(in_features=32, out_features=32, bias=True)
        (proj_q): Linear(in_features=32, out_features=32, bias=True)
        (memory): Sequential(
          (0): Linear(in_features=32, out_features=64, bias=True)
          (1): SiLU()
          (2): Linear(in_features=64, out_features=32, bias=True)
        )
        (gate_theta): Linear(in_features=32, out_features=1, bias=True)
        (gate_eta): Linear(in_features=32, out_features=1, bias=True)
        (gate_alpha): Linear(in_features=32, out_features=1, bias=True)
        (dropout_layer): Dropout(p=0, inplace=False)
      )
      (relu)

Epoch 0 / 50:   0%|          | 0/240 [00:00<?, ?it/s]

--- Train epoch-0, step-240 ---
loss: 0.4881


Evaluation: 100%|██████████| 30/30 [00:10<00:00,  2.77it/s]

--- Eval epoch-0, step-240 ---
roc_auc: 0.5580
pr_auc: 0.1437
accuracy: 0.8795
f1: 0.0000
loss: 0.3714
New best roc_auc score (0.5580) at epoch-0, step-240



...

Epoch 49 / 50:   0%|          | 0/240 [00:00<?, ?it/s]

--- Train epoch-49, step-12000 ---
loss: 0.3534


Evaluation: 100%|██████████| 30/30 [00:10<00:00,  2.87it/s]

--- Eval epoch-49, step-12000 ---
roc_auc: 0.5076
pr_auc: 0.1280
accuracy: 0.8731
f1: 0.0165
loss: 0.4125
Loaded best model



Compute Report: GRASP+LMM
Wall time:          6176.1s (102.93 min)
Epochs:             50
Time per epoch:     123.52s
Throughput:         62.0 samples/s

Energy consumed:    2.655300 kWh
CO2 emissions:      0.977171 kg CO2eq
Energy per epoch:   0.053106 kWh
Energy per sample:  24.9552 J

Peak GPU power:     93 W
Avg GPU power:      93 W
Avg GPU utilization: 11%
Peak GPU memory:    1.73 GB / 150 GB (1%)


In [13]:
grasp_lmm_results = grasp_lmm_trainer.evaluate(test_dataloader)

print("=" * 50)
print("GRASP + LMM — Test Set Performance")
print("=" * 50)
for metric, value in grasp_lmm_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 31/31 [00:11<00:00,  2.65it/s]

GRASP + LMM — Test Set Performance
roc_auc: 0.5597
pr_auc: 0.1454
accuracy: 0.8862
f1: 0.0000
loss: 0.3559


## Step 6: Compare Results — Performance and Compute

Side-by-side comparison of model accuracy AND compute cost. This is the data that matters for understanding the true cost of ML research.

In [14]:
# ── Model Performance ──────────────────────────────────────
print("=" * 60)
print(f"{'MODEL PERFORMANCE':<60}")
print("=" * 60)
print(f"{'Metric':<15} {'LMM Standalone':>15} {'GRASP + LMM':>15}")
print("-" * 60)
for metric in lmm_results:
    lmm_val = lmm_results[metric]
    grasp_val = grasp_lmm_results[metric]
    better = "<-" if grasp_val > lmm_val else "->" if lmm_val > grasp_val else "=="
    print(f"{metric:<15} {lmm_val:>15.4f} {grasp_val:>15.4f}  {better}")

# ── Compute Cost ──────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"{'COMPUTE COST':<60}")
print("=" * 60)
print(f"{'Metric':<25} {'LMM Standalone':>15} {'GRASP + LMM':>15}")
print("-" * 60)

rows = [
    ("Wall time (s)", "wall_time_s", ".1f"),
    ("Time/epoch (s)", "time_per_epoch_s", ".2f"),
    ("Throughput (samples/s)", "batch_throughput_sps", ".1f"),
]

if lmm_compute.get("energy_kwh") is not None:
    rows += [
        ("Energy (kWh)", "energy_kwh", ".6f"),
        ("CO2 (kg)", "co2_kg", ".6f"),
        ("Energy/epoch (kWh)", "energy_per_epoch_kwh", ".6f"),
        ("Energy/sample (J)", "energy_per_sample_j", ".4f"),
    ]

if lmm_compute.get("peak_power_w"):
    rows += [
        ("Peak power (W)", "peak_power_w", ".0f"),
        ("Avg power (W)", "avg_power_w", ".0f"),
        ("GPU utilization (%)", "avg_gpu_util_pct", ".0f"),
    ]

if lmm_compute.get("peak_gpu_mem_gb"):
    rows.append(("Peak GPU memory (GB)", "peak_gpu_mem_gb", ".2f"))

for label, key, fmt in rows:
    lv = lmm_compute.get(key)
    gv = grasp_compute.get(key)
    lstr = f"{lv:{fmt}}" if lv is not None else "N/A"
    gstr = f"{gv:{fmt}}" if gv is not None else "N/A"
    print(f"{label:<25} {lstr:>15} {gstr:>15}")

print("=" * 60)

MODEL PERFORMANCE                                           
Metric           LMM Standalone     GRASP + LMM
------------------------------------------------------------
roc_auc                  0.5212          0.5597  <-
pr_auc                   0.1248          0.1454  <-
accuracy                 0.8852          0.8862  <-
f1                       0.0000          0.0000  ==
loss                     0.3543          0.3559  <-

COMPUTE COST                                                
Metric                     LMM Standalone     GRASP + LMM
------------------------------------------------------------
Wall time (s)                      6231.5          6176.1
Time/epoch (s)                     124.63          123.52
Throughput (samples/s)               61.5            62.0
Energy (kWh)                     2.678351        2.655300
CO2 (kg)                         0.985654        0.977171
Energy/epoch (kWh)               0.053567        0.053106
Energy/sample (J)                 25.1718

## Step 7: Extract Patient Embeddings

Both models produce patient embeddings that can be used for downstream tasks like patient similarity search or cohort discovery.

In [15]:
import torch

grasp_lmm_model.eval()
test_batch = next(iter(test_dataloader))
test_batch["embed"] = True

with torch.no_grad():
    output = grasp_lmm_model(**test_batch)

print(f"Embedding shape: {output['embed'].shape}")
print(f"  - Batch size: {output['embed'].shape[0]}")
print(f"  - Embedding dim: {output['embed'].shape[1]}")

print("\nSample Predictions:")
predictions = output["y_prob"].cpu().numpy()
true_labels = output["y_true"].cpu().numpy()

for i in range(min(5, len(predictions))):
    pred = predictions[i][0]
    true = int(true_labels[i][0])
    label = "Mortality" if pred > 0.5 else "Survival"
    print(f"  Patient {i + 1}: Predicted={pred:.3f}, True={true}, -> {label}")

Embedding shape: torch.Size([32, 96])
  - Batch size: 32
  - Embedding dim: 96

Sample Predictions:
  Patient 1: Predicted=0.142, True=0, -> Survival
  Patient 2: Predicted=0.120, True=0, -> Survival
  Patient 3: Predicted=0.136, True=0, -> Survival
  Patient 4: Predicted=0.132, True=0, -> Survival
  Patient 5: Predicted=0.140, True=0, -> Survival


## Step 8: Compute Cost Summary for Policy

This section translates raw compute metrics into policy-relevant context. The goal is to answer: **what is the real environmental cost of this ML research?**

In [16]:
import json
from datetime import datetime

# ── Policy Context ────────────────────────────────────────
# Reference: average US home uses ~30 kWh/day
# Reference: driving 1 mile ≈ 0.4 kg CO2
# Reference: PUE (datacenter overhead) typically 1.1-1.6x

total_energy = (lmm_compute.get("energy_kwh") or 0) + (grasp_compute.get("energy_kwh") or 0)
total_co2 = (lmm_compute.get("co2_kg") or 0) + (grasp_compute.get("co2_kg") or 0)
total_time = lmm_compute["wall_time_s"] + grasp_compute["wall_time_s"]

print("=" * 60)
print("ENVIRONMENTAL IMPACT — POLICY SUMMARY")
print("=" * 60)

print(f"\n--- This notebook's total footprint ---")
print(f"Total wall time:      {total_time:.0f}s ({total_time/60:.1f} min)")
if total_energy > 0:
    print(f"Total energy:         {total_energy:.6f} kWh")
    print(f"Total CO2:            {total_co2:.6f} kg CO2eq")

    # Analogies
    lightbulb_min = (total_energy / 0.01) * 60  # 10W bulb = 0.01 kWh/hr
    home_fraction = total_energy / 30 * 100
    miles_driven = total_co2 / 0.4

    print(f"\n--- In everyday terms ---")
    print(f"Equivalent to running a 10W LED bulb for {lightbulb_min:.0f} minutes")
    print(f"Equivalent to {home_fraction:.4f}% of a US home's daily electricity")
    if total_co2 > 0:
        print(f"Equivalent to driving {miles_driven:.3f} miles")

    # Extrapolation to full sweep
    FULL_SWEEP_CONFIGS = 108  # from sweep_grasp.py
    sweep_energy = total_energy * FULL_SWEEP_CONFIGS / 2  # /2 because we ran 2 configs
    sweep_co2 = total_co2 * FULL_SWEEP_CONFIGS / 2
    print(f"\n--- Extrapolated: full hyperparameter sweep ({FULL_SWEEP_CONFIGS} configs) ---")
    print(f"Estimated energy:     {sweep_energy:.4f} kWh")
    print(f"Estimated CO2:        {sweep_co2:.4f} kg CO2eq")
    print(f"Equivalent to driving {sweep_co2 / 0.4:.1f} miles")

# GPU efficiency
if lmm_compute.get("avg_gpu_util_pct") is not None:
    avg_util = (lmm_compute["avg_gpu_util_pct"] + grasp_compute.get("avg_gpu_util_pct", 0)) / 2
    print(f"\n--- GPU efficiency ---")
    print(f"Average GPU utilization: {avg_util:.0f}%")
    print(f"Idle energy waste:       ~{(100 - avg_util):.0f}% of GPU power went to idle/waiting")
    if lmm_compute.get("peak_gpu_mem_gb"):
        mem_used = max(lmm_compute.get("peak_gpu_mem_gb", 0), grasp_compute.get("peak_gpu_mem_gb", 0))
        total_gb = gpu_mem_gb if gpu_available else 80
        print(f"Memory utilization:      {mem_used:.1f} GB used of {total_gb:.0f} GB ({mem_used/total_gb*100:.0f}%)")
        if mem_used < total_gb * 0.1:
            print(f"  -> This model could run on a much smaller GPU (< {max(4, mem_used * 2):.0f} GB)")
            print(f"     An H200 (700W TDP) is overkill — a consumer GPU (150W) saves ~4x energy")

print("=" * 60)

ENVIRONMENTAL IMPACT — POLICY SUMMARY

--- This notebook's total footprint ---
Total wall time:      12408s (206.8 min)
Total energy:         5.333652 kWh
Total CO2:            1.962825 kg CO2eq

--- In everyday terms ---
Equivalent to running a 10W LED bulb for 32002 minutes
Equivalent to 17.7788% of a US home's daily electricity
Equivalent to driving 4.907 miles

--- Extrapolated: full hyperparameter sweep (108 configs) ---
Estimated energy:     288.0172 kWh
Estimated CO2:        105.9925 kg CO2eq
Equivalent to driving 265.0 miles

--- GPU efficiency ---
Average GPU utilization: 11%
Idle energy waste:       ~89% of GPU power went to idle/waiting
Memory utilization:      1.7 GB used of 150 GB (1%)
  -> This model could run on a much smaller GPU (< 4 GB)
     An H200 (700W TDP) is overkill — a consumer GPU (150W) saves ~4x energy
